# Smart Retail Intelligence System
## Notebook 2: Product Recommendations and Search

| Part | What we do | Syllabus |
|---|---|---|
| 1 | Find products that are bought together (Apriori) | Module 4 |
| 2 | Find paths between products with BFS, DFS, UCS and A* | Module 2 |

**Run Notebook 1 first.** It saves the two files this notebook loads (`basket.pkl` and `product_names.pkl`).

In [ ]:
# Step 1: connect Google Drive and go to the project folder (run this cell first)
!pip install -q mlxtend
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir('/content/drive/MyDrive/aiml_minproject')
print(os.listdir('.'))

In [ ]:
# Step 2: import the libraries and load the files saved by Notebook 1
import warnings
warnings.filterwarnings("ignore")

import random
import pandas as pd
import numpy as np

basket = pd.read_pickle("basket.pkl")                  # rows = invoices, columns = products (True / False)
product_names = pd.read_pickle("product_names.pkl")    # product code -> product name

def get_name(code):
    return product_names.get(code, code)

print("Number of invoices:", basket.shape[0], "| number of products:", basket.shape[1])

## Part 1: Which products are bought together? (Apriori)
A rule like **Mug -> Spoon** means "people who buy a mug often buy a spoon too".
- **Support:** how many invoices contain both products
- **Confidence:** of the invoices with the mug, how many also contain the spoon
- **Lift:** how much more often they are bought together than by chance (above 1 means a real link)

In [ ]:
from mlxtend.frequent_patterns import apriori, association_rules

# Step 1: find products that appear together in at least 2% of invoices
frequent = apriori(basket, min_support=0.02, use_colnames=True)

# Step 2: turn them into rules and keep rules with lift above 1
try:
    rules = association_rules(frequent, metric="lift", min_threshold=1.0)
except TypeError:      # newer versions of mlxtend need one more argument
    rules = association_rules(frequent, num_itemsets=len(basket), metric="lift", min_threshold=1.0)

# Step 3: keep only simple rules (one product -> one product), strongest first
rules = rules[(rules["antecedents"].apply(len) == 1) & (rules["consequents"].apply(len) == 1)]
rules = rules.sort_values("lift", ascending=False).reset_index(drop=True)

# Step 4: add product names so the table is easy to read
if_bought = []
also_bought = []
for i in range(len(rules)):
    if_bought.append(get_name(list(rules["antecedents"].iloc[i])[0]))
    also_bought.append(get_name(list(rules["consequents"].iloc[i])[0]))
rules["If bought"] = if_bought
rules["Also bought"] = also_bought

print("Number of rules:", len(rules))
rules[["If bought", "Also bought", "support", "confidence", "lift"]].head(10).round(3)

### A simple recommender
Give it a product, and it returns the products with the strongest rules.

In [ ]:
def recommend(product_code, how_many=5):
    matches = []
    for i in range(len(rules)):
        if list(rules["antecedents"].iloc[i])[0] == product_code:
            matches.append(i)
    result = rules.iloc[matches].sort_values("lift", ascending=False).head(how_many)
    return result[["Also bought", "confidence", "lift"]].round(2)

example_code = list(rules["antecedents"].iloc[0])[0]
print("If the customer buys:", get_name(example_code))
recommend(example_code)

## Part 2: Search algorithms on a product graph
We turn the rules into a **graph**:
- each **node** is a product
- each **edge** joins two products that are bought together
- the **cost** of an edge is `1 / lift`, so strongly linked products are "close" (low cost)

**Task:** find a path from one product to another, and compare the algorithms.

| Algorithm | Idea | Cheapest path guaranteed? |
|---|---|---|
| BFS | Look at all neighbours first, then their neighbours | No |
| DFS | Follow one path as deep as possible, then go back | No |
| UCS | Always extend the cheapest path found so far | Yes |
| A* | Cheapest path so far **plus** a guess of the cost still left | Yes |

**Our guess for A\*:** (number of edges still to go) x (cheapest edge in the graph). The real cost can never be lower than this, so A* stays correct.

In [ ]:
# Build the graph: graph[product][neighbour] = cost of the edge
graph = {}
for i in range(len(rules)):
    a = list(rules["antecedents"].iloc[i])[0]
    b = list(rules["consequents"].iloc[i])[0]
    cost = 1 / rules["lift"].iloc[i]
    if a not in graph:
        graph[a] = {}
    if b not in graph:
        graph[b] = {}
    graph[a][b] = cost
    graph[b][a] = cost
print("Graph has", len(graph), "products")

# The cheapest edge in the graph (used for the A* guess)
all_costs = []
for a in graph:
    for b in graph[a]:
        all_costs.append(graph[a][b])
cheapest_edge = min(all_costs)

def path_cost(path):
    total = 0
    for i in range(len(path) - 1):
        total = total + graph[path[i]][path[i + 1]]
    return total

def edges_from(start):
    # how many edges it takes to reach every product from "start"
    distance = {start: 0}
    queue = [start]
    while len(queue) > 0:
        node = queue.pop(0)
        for neighbour in graph[node]:
            if neighbour not in distance:
                distance[neighbour] = distance[node] + 1
                queue.append(neighbour)
    return distance

In [ ]:
# Each function returns the path it found and the number of nodes it looked at ("expanded")

def bfs(start, goal):
    queue = [[start]]                  # a list of paths: first in, first out
    visited = [start]
    expanded = 0
    while len(queue) > 0:
        path = queue.pop(0)
        node = path[-1]
        expanded = expanded + 1
        if node == goal:
            return path, expanded
        for neighbour in sorted(graph[node]):
            if neighbour not in visited:
                visited.append(neighbour)
                queue.append(path + [neighbour])

def dfs(start, goal):
    stack = [[start]]                  # a list of paths: last in, first out
    visited = []
    expanded = 0
    while len(stack) > 0:
        path = stack.pop()
        node = path[-1]
        if node in visited:
            continue
        visited.append(node)
        expanded = expanded + 1
        if node == goal:
            return path, expanded
        for neighbour in sorted(graph[node], reverse=True):
            if neighbour not in visited:
                stack.append(path + [neighbour])

def ucs(start, goal):
    open_list = [[0, [start]]]         # each item is [cost so far, path]
    visited = []
    expanded = 0
    while len(open_list) > 0:
        open_list.sort()               # cheapest path first
        cost, path = open_list.pop(0)
        node = path[-1]
        if node in visited:
            continue
        visited.append(node)
        expanded = expanded + 1
        if node == goal:
            return path, expanded
        for neighbour in graph[node]:
            if neighbour not in visited:
                open_list.append([cost + graph[node][neighbour], path + [neighbour]])

def astar(start, goal):
    edges_left = edges_from(goal)      # edges still to go, from every product to the goal
    open_list = [[edges_left[start] * cheapest_edge, 0, [start]]]   # [cost so far + guess, cost so far, path]
    visited = []
    expanded = 0
    while len(open_list) > 0:
        open_list.sort()
        total, cost, path = open_list.pop(0)
        node = path[-1]
        if node in visited:
            continue
        visited.append(node)
        expanded = expanded + 1
        if node == goal:
            return path, expanded
        for neighbour in graph[node]:
            if neighbour not in visited:
                new_cost = cost + graph[node][neighbour]
                guess = edges_left[neighbour] * cheapest_edge
                open_list.append([new_cost + guess, new_cost, path + [neighbour]])

### Try the algorithms
We pick 5 random pairs of products that are at least 2 edges apart, run all four algorithms on each pair, and compare the **path cost** (lower is better) and the **nodes expanded** (fewer means a faster search).

In [ ]:
# Pick 5 pairs of products that are connected and at least 2 edges apart
random.seed(1)
products = list(graph.keys())
pairs = []
tries = 0
while len(pairs) < 5 and tries < 5000:
    tries = tries + 1
    start, goal = random.sample(products, 2)
    distance = edges_from(start)
    if goal in distance and distance[goal] >= 2 and (start, goal) not in pairs:
        pairs.append((start, goal))

# Show the first pair in detail
start, goal = pairs[0]
print("From:", get_name(start))
print("To:  ", get_name(goal))
algorithms = [("BFS", bfs), ("DFS", dfs), ("UCS", ucs), ("A*", astar)]
for name, function in algorithms:
    path, expanded = function(start, goal)
    print("\n" + name + ": cost =", round(path_cost(path), 3), "| nodes expanded =", expanded)
    print("   " + " -> ".join(get_name(p) for p in path))

In [ ]:
# Run every algorithm on every pair and collect the results
results = []
for start, goal in pairs:
    for name, function in algorithms:
        path, expanded = function(start, goal)
        results.append({"Algorithm": name,
                        "Cost": path_cost(path),
                        "Edges": len(path) - 1,
                        "Nodes expanded": expanded})

table = pd.DataFrame(results)
table.groupby("Algorithm")[["Cost", "Edges", "Nodes expanded"]].mean().round(3)

**How to read the table (averages over the 5 pairs)**
- **UCS and A*** have the lowest cost, because they always find the cheapest path.
- **A*** usually expands fewer nodes than UCS, because its guess points the search towards the goal.
- **BFS** finds the path with the fewest edges, but not always the cheapest.
- **DFS** often takes a long, expensive route.

## Summary
- **Apriori** found products that are bought together, such as matching sets and variants of the same item.
- Four search algorithms were compared on a graph built from those rules.

**Limitations:** the rules use only the 150 most-bought products, and some products in the graph are not connected to each other.